# 🍄 YOLOv8 Mushroom Object Detection
**Full pipeline:** Dataset prep → Training → Performance Metrics → Confusion Matrix → Sample Predictions

---
> **Before running:** Go to `Runtime → Change runtime type → GPU (T4)` for fast training.

## ⚙️ Cell 1 — Install dependencies

In [ ]:
!pip install ultralytics -q
!pip install seaborn scikit-learn -q
import ultralytics
ultralytics.checks()
print('✅ Dependencies ready')

## 📂 Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

## 🗂️ Cell 3 — Configure dataset path

> **Edit `DATASET_ROOT` below** to match your Google Drive folder path.
> Your folder should have: `mushroom dataset/images/` and `mushroom dataset/labels/`

In [ ]:
import os

# ─── EDIT THIS PATH ───────────────────────────────────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/mushroom dataset'   # <-- change if needed
# ──────────────────────────────────────────────────────────────────────────────

# Class names — update if your labels have real names
CLASS_NAMES = None   # e.g. ['Amanita', 'Boletus', 'Chanterelle'] or leave None to auto-detect

# Training config
MODEL      = 'yolov8n.pt'   # yolov8n (fast) / yolov8s / yolov8m / yolov8l / yolov8x (accurate)
EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 16

SRC_IMAGES = os.path.join(DATASET_ROOT, 'images')
SRC_LABELS = os.path.join(DATASET_ROOT, 'labels')

assert os.path.exists(SRC_IMAGES), f'Images folder not found: {SRC_IMAGES}'
assert os.path.exists(SRC_LABELS), f'Labels folder not found: {SRC_LABELS}'
print(f'✅ Found images : {len(os.listdir(SRC_IMAGES))} files')
print(f'✅ Found labels : {len(os.listdir(SRC_LABELS))} files')

## 📁 Cell 4 — Build YOLOv8 folder structure (train / val / test split)

In [ ]:
import shutil, random, yaml
from pathlib import Path

random.seed(42)
OUT = Path('/content/mushroom_yolo')

img_exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
all_imgs = [f for f in Path(SRC_IMAGES).iterdir() if f.suffix.lower() in img_exts]

paired = []
for img in all_imgs:
    lbl = Path(SRC_LABELS) / (img.stem + '.txt')
    if lbl.exists():
        paired.append((img, lbl))

random.shuffle(paired)
n = len(paired)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)
splits  = {'train': paired[:n_train],
            'val'  : paired[n_train:n_train+n_val],
            'test' : paired[n_train+n_val:]}

for split, items in splits.items():
    for sub in ['images','labels']:
        (OUT / split / sub).mkdir(parents=True, exist_ok=True)
    for img_p, lbl_p in items:
        shutil.copy2(img_p, OUT / split / 'images' / img_p.name)
        shutil.copy2(lbl_p, OUT / split / 'labels' / lbl_p.name)
    print(f'  {split:>5}: {len(items)} samples')

# Auto-detect classes
if CLASS_NAMES is None:
    class_ids = set()
    for img_p, lbl_p in paired[:500]:
        with open(lbl_p) as f:
            for line in f:
                parts = line.strip().split()
                if parts: class_ids.add(int(parts[0]))
    CLASS_NAMES = [f'class_{i}' for i in sorted(class_ids)]
    print(f'  Auto-detected {len(CLASS_NAMES)} classes: {CLASS_NAMES}')

yaml_data = {
    'path' : str(OUT.resolve()),
    'train': 'train/images',
    'val'  : 'val/images',
    'test' : 'test/images',
    'nc'   : len(CLASS_NAMES),
    'names': CLASS_NAMES
}
YAML_PATH = str(OUT / 'dataset.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f'\n✅ Dataset structured → {OUT}')
print(f'✅ dataset.yaml → {YAML_PATH}')

## 🚀 Cell 5 — Train YOLOv8
> Training runs for **50 epochs**. You'll see live loss/mAP updates each epoch.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data     = YAML_PATH,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    project  = '/content/mushroom_runs',
    name     = 'train',
    exist_ok = True,
    plots    = True,
    verbose  = True,
)

BEST_MODEL_PATH = '/content/mushroom_runs/train/weights/best.pt'
print(f'\n✅ Training complete!')
print(f'✅ Best model saved → {BEST_MODEL_PATH}')

## 📊 Cell 6 — Performance Metrics (mAP, Precision, Recall, F1)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Load best model and run validation
best_model = YOLO(BEST_MODEL_PATH)
metrics = best_model.val(data=YAML_PATH, split='val', plots=True, verbose=False)

map50    = metrics.box.map50
map5095  = metrics.box.map
prec     = metrics.box.mp
rec      = metrics.box.mr
f1       = 2 * prec * rec / (prec + rec + 1e-9)

print('\n' + '='*50)
print('       PERFORMANCE METRICS SUMMARY')
print('='*50)
print(f'  mAP@50       : {map50:.4f}')
print(f'  mAP@50-95    : {map5095:.4f}')
print(f'  Precision    : {prec:.4f}')
print(f'  Recall       : {rec:.4f}')
print(f'  F1-Score     : {f1:.4f}')
print('='*50)

# ── Plot metrics bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('YOLOv8 — Performance Metrics', fontsize=15, fontweight='bold')

# Overall metrics
labels_ov = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'F1-Score']
vals_ov   = [map50, map5095, prec, rec, f1]
colors_ov = ['#2196F3','#673AB7','#4CAF50','#FF9800','#F44336']
bars = axes[0].bar(labels_ov, vals_ov, color=colors_ov, edgecolor='white', linewidth=0.5, width=0.5)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Overall metrics', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, vals_ov):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Per-class mAP (if available)
try:
    per_class_ap = metrics.box.ap50
    cls_labels   = CLASS_NAMES[:len(per_class_ap)]
    axes[1].barh(cls_labels, per_class_ap, color='#2196F3', edgecolor='white', linewidth=0.5)
    axes[1].set_xlim(0, 1.1)
    axes[1].set_title('mAP@50 per class', fontsize=12)
    axes[1].set_xlabel('AP@50')
    axes[1].grid(axis='x', alpha=0.3)
    for i, val in enumerate(per_class_ap):
        axes[1].text(val+0.01, i, f'{val:.3f}', va='center', fontsize=9)
except Exception:
    axes[1].text(0.5, 0.5, 'Per-class AP\nnot available', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=12, color='gray')
    axes[1].set_title('mAP@50 per class', fontsize=12)

plt.tight_layout()
plt.savefig('/content/performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/performance_metrics.png')

## 📉 Cell 7 — Training Curves (Loss + mAP over Epochs)

In [ ]:
results_csv = '/content/mushroom_runs/train/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Training Curves — Mushroom Detection', fontsize=15, fontweight='bold')

plots = [
    ('train/box_loss',         'Train Box Loss',   '#F44336'),
    ('train/cls_loss',         'Train Class Loss', '#FF9800'),
    ('val/box_loss',           'Val Box Loss',     '#9C27B0'),
    ('metrics/precision(B)',   'Precision',        '#4CAF50'),
    ('metrics/recall(B)',      'Recall',           '#2196F3'),
    ('metrics/mAP50(B)',       'mAP@50',           '#009688'),
]

for ax, (col, title, color) in zip(axes.flatten(), plots):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=2)
        ax.fill_between(df['epoch'], df[col], alpha=0.1, color=color)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=1)
    else:
        ax.text(0.5,0.5,f'{title}\n(column not found)', ha='center',va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(title)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/training_curves.png')

## 🔲 Cell 8 — Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from pathlib import Path

test_img_dir = OUT / 'test' / 'images'
test_lbl_dir = OUT / 'test' / 'labels'

y_true, y_pred = [], []
test_images = list(test_img_dir.iterdir())
print(f'Running predictions on {min(300, len(test_images))} test images...')

for img_file in test_images[:300]:
    lbl_file = test_lbl_dir / (img_file.stem + '.txt')
    if not lbl_file.exists():
        continue
    with open(lbl_file) as f:
        gt = [int(l.split()[0]) for l in f if l.strip()]
    if not gt:
        continue
    result = best_model(str(img_file), verbose=False)[0]
    preds  = [int(c) for c in result.boxes.cls.cpu().numpy()] if result.boxes and len(result.boxes) else []
    y_true.append(max(set(gt), key=gt.count))
    y_pred.append(max(set(preds), key=preds.count) if preds else -1)

valid   = [(t,p) for t,p in zip(y_true,y_pred) if p != -1]
yt, yp  = zip(*valid) if valid else ([],[]) 
labels  = sorted(set(yt)|set(yp))
lnames  = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in labels]

cm      = confusion_matrix(yt, yp, labels=labels)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Confusion Matrix — Mushroom Detection', fontsize=15, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=lnames, yticklabels=lnames, ax=axes[0],
            linewidths=0.5, linecolor='white')
axes[0].set_title('Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted Class', fontsize=11)
axes[0].set_ylabel('True Class', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=lnames, yticklabels=lnames, ax=axes[1],
            linewidths=0.5, linecolor='white', vmin=0, vmax=1)
axes[1].set_title('Normalized (Recall per class)', fontsize=12)
axes[1].set_xlabel('Predicted Class', fontsize=11)
axes[1].set_ylabel('True Class', fontsize=11)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/confusion_matrix.png')

## 🖼️ Cell 9 — Sample Prediction Images (with Bounding Boxes)
> Shows 12 test images with predicted bounding boxes, class labels, and confidence scores drawn on them.

In [ ]:
import cv2, random
from pathlib import Path

test_imgs = list((OUT / 'test' / 'images').iterdir())
random.shuffle(test_imgs)
samples = test_imgs[:12]

COLORS = [
    (46,204,113),(52,152,219),(231,76,60),(155,89,182),
    (241,196,15),(26,188,156),(230,126,34),(149,165,166),
    (52,73,94),(192,57,43),(39,174,96),(41,128,185)
]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('YOLOv8 — Sample Predictions on Test Images', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, img_path in enumerate(samples):
    result  = best_model(str(img_path), verbose=False)[0]
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    if result.boxes and len(result.boxes):
        for box in result.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            conf   = float(box.conf[0].cpu().numpy())
            color  = COLORS[cls_id % len(COLORS)]
            name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            label  = f'{name}  {conf:.2f}'

            cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(img_rgb, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
            cv2.putText(img_rgb, label, (x1+2, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1, cv2.LINE_AA)

    axes[idx].imshow(img_rgb)
    n_det = len(result.boxes) if result.boxes else 0
    axes[idx].set_title(f'{img_path.name}\n{n_det} detection(s)', fontsize=9)
    axes[idx].axis('off')

for j in range(len(samples), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: /content/sample_predictions.png')

## 📋 Cell 10 — Full Evaluation Report (PR Curve, F1 Curve, per-class table)

In [ ]:
from IPython.display import Image, display
import os

val_dir = '/content/mushroom_runs/train'

# Show YOLOv8 auto-generated plots
auto_plots = [
    ('results.png',           'Training Results Summary'),
    ('confusion_matrix.png',  'YOLOv8 Confusion Matrix (auto)'),
    ('PR_curve.png',          'Precision-Recall Curve'),
    ('F1_curve.png',          'F1-Confidence Curve'),
    ('P_curve.png',           'Precision-Confidence Curve'),
    ('R_curve.png',           'Recall-Confidence Curve'),
]

for fname, title in auto_plots:
    fpath = os.path.join(val_dir, fname)
    if os.path.exists(fpath):
        print(f'\n── {title} ──')
        display(Image(fpath, width=800))
    else:
        print(f'  (not found: {fpath})')

# Per-class metrics table
print('\n── Per-Class Metrics Table ──')
try:
    ap50_per   = metrics.box.ap50
    ap5095_per = metrics.box.ap
    rows = []
    for i, (a50, a95) in enumerate(zip(ap50_per, ap5095_per)):
        cname = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f'class_{i}'
        rows.append({'Class': cname, 'AP@50': f'{a50:.4f}', 'AP@50-95': f'{a95:.4f}'})
    df_cls = pd.DataFrame(rows)
    print(df_cls.to_string(index=False))
    df_cls.to_csv('/content/per_class_metrics.csv', index=False)
    print('\n✅ Saved: /content/per_class_metrics.csv')
except Exception as e:
    print(f'  Per-class table unavailable: {e}')

## 💾 Cell 11 — Save everything back to Google Drive

In [ ]:
import shutil

# ─── EDIT: where on your Drive to save ────────────────────────────────────────
DRIVE_SAVE = '/content/drive/MyDrive/mushroom_yolo_results'
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_SAVE, exist_ok=True)

files_to_save = [
    ('/content/performance_metrics.png',   'performance_metrics.png'),
    ('/content/training_curves.png',       'training_curves.png'),
    ('/content/confusion_matrix.png',      'confusion_matrix.png'),
    ('/content/sample_predictions.png',    'sample_predictions.png'),
    ('/content/per_class_metrics.csv',     'per_class_metrics.csv'),
    (BEST_MODEL_PATH,                       'best.pt'),
]

# Also copy YOLOv8 auto plots
for fname, _ in auto_plots:
    fpath = os.path.join(val_dir, fname)
    if os.path.exists(fpath):
        files_to_save.append((fpath, fname))

for src, dst_name in files_to_save:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_SAVE, dst_name))
        print(f'  ✅ Saved → {dst_name}')
    else:
        print(f'  ⚠️  Not found: {src}')

print(f'\n🎉 All outputs saved to Google Drive: {DRIVE_SAVE}')

## 🔍 Cell 12 — Predict on a single new image

In [ ]:
# Change this to any image path on your Drive or Colab
TEST_IMAGE = '/content/drive/MyDrive/mushroom dataset/images/your_image.jpg'

result = best_model(TEST_IMAGE, verbose=False)[0]
img    = cv2.cvtColor(cv2.imread(TEST_IMAGE), cv2.COLOR_BGR2RGB)

for box in (result.boxes or []):
    x1,y1,x2,y2 = map(int, box.xyxy[0].cpu().numpy())
    cls_id = int(box.cls[0].cpu().numpy())
    conf   = float(box.conf[0].cpu().numpy())
    name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
    color  = COLORS[cls_id % len(COLORS)]
    cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
    cv2.putText(img,f'{name} {conf:.2f}',(x1,y1-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2)

plt.figure(figsize=(10,8))
plt.imshow(img)
plt.axis('off')
plt.title('Single image prediction', fontsize=13)
plt.tight_layout()
plt.show()